## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
date = '20250212'

In [5]:
## base

base_query = f"""

    select 
        yyyymmdd,
        city_name,
        service_obj_service_name service_name,
        customer_id,
        order_id,
        order_status,
        modified_order_status,
        
        customer_requested_latitude,
        customer_requested_longitude,
        customer_requested_hex_8,
        customer_requested_hex_10,
        customer_requested_hex_12,
        
        customer_cancelled_latitude,
        customer_cancelled_longitude,
        
        captain_accepted_latitude,
        captain_accepted_longitude,
        
        captain_arrived_latitude,
        captain_arrived_longitude,
        
        captain_started_latitude,
        captain_started_longitude
    
    from 
        reports.sql_ingestion_platform_olf_test_view
    where 
        yyyymmdd = '{date}'
        and service_obj_service_name IN ('Auto', 'Link', 'CabEconomy', 'Bike Lite', 'Bike Metro', 'CabPremium', 'C2C', 'E rickshaw', 'Cab SUV', 'Auto Pool', 'Auto NCR', 'Bike Pink', 'Auto C2C', 'Auto Pet', 'AutoPremium')
"""

df_base_query = pd.read_sql(base_query, connection)
df_base_query

,yyyymmdd,city_name,service_name,customer_id,order_id,order_status,modified_order_status,customer_requested_latitude,customer_requested_longitude,customer_requested_hex_8,customer_requested_hex_10,customer_requested_hex_12,customer_cancelled_latitude,customer_cancelled_longitude,captain_accepted_latitude,captain_accepted_longitude,captain_arrived_latitude,captain_arrived_longitude,captain_started_latitude,captain_started_longitude
0,20250212,Hyderabad,Bike Lite,630df66f5843ead84db5a584,67acaf7da3208819af37cce4,dropped,dropped,17.468583,78.506903,8860b1964bfffff,8a60b1964b8ffff,8c60b1964b8d7ff,NaN,NaN,17.467592,78.506874,17.468500,78.507294,17.468500,78.507290
1,20250212,Hyderabad,Bike Lite,62b93d8521385b1fad55c328,67ac54ce8aba3a1a3dd639ba,customerCancelled,COBRA,17.594208,78.451764,8860b195ebfffff,8a60b195ea17fff,8c60b195ea1edff,17.594203,78.451744,NaN,NaN,NaN,NaN,NaN,NaN
2,20250212,Hyderabad,Auto,637d40921ed9b8aa20b4a827,67acaf7f716fb34a65360125,dropped,dropped,17.405175,78.539239,8860b52eb5fffff,8a60b52ebcdffff,8c60b52ebcdb7ff,NaN,NaN,17.397930,78.536018,17.405172,78.539101,17.405173,78.539100
3,20250212,Hyderabad,Auto,66cd9fd2d363fd861bd3cd5a,67ac54cfd8892a588c035fa5,dropped,dropped,17.414236,78.412411,8860a259cbfffff,8a60a259ca0ffff,8c60a259ca089ff,NaN,NaN,17.416069,78.412102,17.414261,78.415077,17.414269,78.415113
4,20250212,Bangalore,Link,63a79fbdccd02684835d7306,67acaf82a264e77c0ecf96c8,dropped,dropped,12.896963,77.567873,8861892487fffff,8a618924861ffff,8c618924861cdff,NaN,NaN,12.883908,77.541145,12.884446,77.552979,12.884446,77.552979
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5023520,20250212,Hyderabad,Auto,5cce41c6f4a135067a0f327a,67aca532ec00ca66b811ad24,dropped,dropped,17.400120,78.561063,8860b52ea9fffff,8a60b52ea90ffff,8c60b52ea90bdff,NaN,NaN,17.399178,78.560600,17.400070,78.560394,17.400069,78.560390
5023521,20250212,Hyderabad,Link,63df97fa99ac6426fd69daee,67aca534ae02565284796ab6,dropped,dropped,17.427099,78.341149,8860a24a09fffff,8a60a24a084ffff,8c60a24a084e5ff,NaN,NaN,17.427217,78.341209,17.427217,78.341209,17.427218,78.341207
5023522,20250212,Delhi,Bike Lite,634d53716f69fe87d22f2c46,67aca5368e45b947b2537699,customerCancelled,COBRA,28.450668,77.071944,883da11abdfffff,8a3da11abd77fff,8c3da11abd761ff,28.450701,77.072023,NaN,NaN,NaN,NaN,NaN,NaN
5023523,20250212,Bangalore,Auto,63170f0f00f4d737f2cb1455,67aca5384b997b3b4fdd8462,customerCancelled,OCARA,12.914737,77.625783,8861892423fffff,8a618925c927fff,8c618925c925bff,12.914701,77.625834,12.910246,77.632462,NaN,NaN,NaN,NaN


In [6]:
df_base_query.shape

(5023525, 20)

In [7]:
df_base_query.to_parquet('/Users/E2074/local-datasets/customer/qc/sql_ingestion_platform_olf_test_view.parquet', index=False)

In [12]:
# df_base_query = pd.read_parquet('/Users/E2074/local-datasets/customer/qc/sql_ingestion_platform_olf_test_view.parquet')

In [18]:
## order_requested

order_requested = f"""


select 
    yyyymmdd,
    city_name,
    service_obj_service_name service_name,
    customer_id,
    order_id,
    order_status,
    event_type,
    
    customer_location_latitude customer_requested_latitude,
    customer_location_latitude customer_requested_longitude,
    customer_location_hex_8 customer_requested_hex_8,
    customer_location_hex_10 customer_requested_hex_10,
    customer_location_hex_12 customer_requested_hex_12 
    
from 
    orders.order_logs_immutable
where 
    yyyymmdd = '{date}'
    and service_obj_service_name IN ('Auto', 'Link', 'CabEconomy', 'Bike Lite', 'Bike Metro', 'CabPremium', 'C2C', 'E rickshaw', 'Cab SUV', 'Auto Pool', 'Auto NCR', 'Bike Pink', 'Auto C2C', 'Auto Pet', 'AutoPremium')
    and event_type = 'order_requested'
"""

df_order_requested = pd.read_sql(order_requested, connection)
df_base_query

DatabaseError: {'message': 'Query exceeded the maximum execution time limit of 10.00m', 'errorCode': 131075, 'errorName': 'EXCEEDED_TIME_LIMIT', 'errorType': 'INSUFFICIENT_RESOURCES', 'failureInfo': {'type': 'io.trino.spi.TrinoException', 'message': 'Query exceeded the maximum execution time limit of 10.00m', 'suppressed': [], 'stack': ['io.trino.execution.QueryTracker.enforceTimeLimits(QueryTracker.java:187)', 'io.trino.execution.QueryTracker.lambda$start$0(QueryTracker.java:89)', 'com.google.common.util.concurrent.MoreExecutors$ScheduledListeningDecorator$NeverSuccessfulListenableFutureTask.run(MoreExecutors.java:730)', 'java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)', 'java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)', 'java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)', 'java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.jav

In [ ]:
df_base_query.to_parquet('fare_estimates_bangalore_link_20241021.parquet', index=False)